In [54]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from dotenv import load_dotenv
import os

load_dotenv("../.env")  


spark = SparkSession.builder \
    .appName("PagilaAnalysis") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

In [55]:

jdbc_url = f"jdbc:postgresql://{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
props = {
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "driver": os.getenv("DB_DRIVER")
}
def load(table):
    return spark.read.jdbc(url=jdbc_url, table=table, properties=props)


film          = load("film")
category      = load("category")
film_category = load("film_category")
actor         = load("actor")
film_actor    = load("film_actor")
inventory     = load("inventory")
rental        = load("rental")
customer      = load("customer")
address       = load("address")
city          = load("city")
payment       = load("payment")


In [56]:

film.select("title").show()

+----------------+
|           title|
+----------------+
|ACADEMY DINOSAUR|
|  ACE GOLDFINGER|
|ADAPTATION HOLES|
|AFFAIR PREJUDICE|
|     AFRICAN EGG|
|    AGENT TRUMAN|
| AIRPLANE SIERRA|
| AIRPORT POLLOCK|
|   ALABAMA DEVIL|
|ALADDIN CALENDAR|
| ALAMO VIDEOTAPE|
|  ALASKA PHANTOM|
|      DATE SPEED|
|     ALI FOREVER|
|  ALICE FANTASIA|
|    ALIEN CENTER|
| ALLEY EVOLUTION|
|      ALONE TRIP|
|   ALTER VICTORY|
|    AMADEUS HOLY|
+----------------+
only showing top 20 rows



In [57]:
#Output the number of movies in each category, sorted in descending order. 

q1 = (
    film_category
    .join(category, "category_id")
    .groupBy(category["name"].alias("category"))
    .agg(F.count("film_id").alias("movie_count"))
    .orderBy(F.desc("movie_count"))
)
q1.show()

+-----------+-----------+
|   category|movie_count|
+-----------+-----------+
|      Drama|        152|
|      Music|        152|
|     Travel|        151|
|    Foreign|        150|
|      Games|        150|
|   Children|        150|
|     Action|        149|
|     Sci-Fi|        149|
|  Animation|        148|
|     Family|        147|
|   Classics|        147|
|        New|        147|
|     Sports|        145|
|Documentary|        145|
|     Comedy|        143|
|     Horror|        142|
+-----------+-----------+



In [58]:
#Output the 10 actors whose movies rented the most, sorted in descending order. 

q2 = (
    film_actor
    .join(actor, "actor_id")
    .join(inventory, "film_id")
    .join(rental, "inventory_id")
    .groupBy("actor_id",
             actor["first_name"].alias("first_name"),
             actor["last_name"].alias("last_name"))
    .agg(F.count("rental_id").alias("rental_count"))
    .orderBy(F.desc("rental_count"))
    .limit(10)
)
q2.show()


+--------+----------+-----------+------------+
|actor_id|first_name|  last_name|rental_count|
+--------+----------+-----------+------------+
|     107|      GINA|  DEGENERES|         753|
|     181|   MATTHEW|     CARREY|         678|
|     198|      MARY|     KEITEL|         674|
|     144|    ANGELA|WITHERSPOON|         654|
|     102|    WALTER|       TORN|         640|
|      60|     HENRY|      BERRY|         612|
|     150|     JAYNE|      NOLTE|         611|
|      37|       VAL|     BOLGER|         605|
|      23|    SANDRA|     KILMER|         604|
|      90|      SEAN|    GUINESS|         599|
+--------+----------+-----------+------------+



In [59]:
#Output the category of movies on which the most money was spent. 

q3 = (
    payment
    .join(rental, "rental_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .groupBy(category["name"].alias("category"))
    .agg(F.sum("amount").alias("total_spent"))
    .orderBy(F.desc("total_spent"))
    .limit(1)
)
q3.show()

+--------+-----------+
|category|total_spent|
+--------+-----------+
| Foreign|   10507.67|
+--------+-----------+



In [60]:
#Output the names of movies that are not in the inventory. 

q4 = (
    film
    .join(inventory, "film_id", "left_anti")
    .select("title")
)
q4.show(truncate=False)

+---------------------+
|title                |
+---------------------+
|CHOCOLATE DUCK       |
|BUTCH PANTHER        |
|VOLUME HOUSE         |
|ORDER BETRAYED       |
|TADPOLE PARK         |
|KILL BROTHERHOOD     |
|FRANKENSTEIN STRANGER|
|CROSSING DIVORCE     |
|SUICIDES SILENCE     |
|CATCH AMISTAD        |
|PERDITION FARGO      |
|FLOATS GARDEN        |
|GUMP DATE            |
|WALLS ARTIST         |
|GLADIATOR WESTWARD   |
|HOCUS FRIDA          |
|ARSENIC INDEPENDENCE |
|MUPPET MILE          |
|FIREHOUSE VIETNAM    |
|ROOF CHAMPION        |
+---------------------+
only showing top 20 rows



In [61]:
#Output the top 3 actors who have appeared most in movies in the “Children” category. 
#If several actors have the same number of movies, output all of them. 
children_actors = (
    film_category
    .join(category, "category_id")
    .filter(category["name"] == "Children")
    .join(film_actor, "film_id")
    .join(actor, "actor_id")
    .groupBy("actor_id",
             actor["first_name"].alias("first_name"),
             actor["last_name"].alias("last_name"))
    .agg(F.count("film_id").alias("movie_count"))
)

window_spec = Window.orderBy(F.desc("movie_count"))

q5 = (
    children_actors
    .withColumn("rnk", F.dense_rank().over(window_spec))
    .filter(F.col("rnk") <= 3)
    .orderBy(F.desc("movie_count"))
)
q5.show()

+--------+----------+---------+-----------+---+
|actor_id|first_name|last_name|movie_count|rnk|
+--------+----------+---------+-----------+---+
|     105|    SIDNEY|    CROWE|          9|  1|
|     139|      EWAN|  GOODING|          9|  1|
|     133|   RICHARD|     PENN|          9|  1|
|      87|   SPENCER|     PECK|          8|  2|
|     145|       KIM|    ALLEN|          8|  2|
|      66|      MARY|    TANDY|          8|  2|
|      29|      ALEC|    WAYNE|          8|  2|
|      56|       DAN|   HARRIS|          8|  2|
|     149|   RUSSELL|   TEMPLE|          8|  2|
|     181|   MATTHEW|   CARREY|          8|  2|
|     131|      JANE|  JACKMAN|          8|  2|
|     142|      JADA|    RYDER|          8|  2|
|      84|     JAMES|     PITT|          7|  3|
|     108|    WARREN|    NOLTE|          7|  3|
|     123|  JULIANNE|    DENCH|          7|  3|
|      34|    AUDREY|  OLIVIER|          7|  3|
|      96|      GENE|   WILLIS|          7|  3|
|      65|    ANGELA|   HUDSON|         

26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 16:45:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/26 1

In [62]:
#!!!!
#Output cities with the number of active and inactive customers (active - customer.active = 1).
#Sort by the number of inactive customers in descending order.
q6 = (
    customer
    .join(address, "address_id")
    .join(city, "city_id")
    .groupBy(city["city"].alias("city"))
    .agg(
        F.sum(F.when(F.col("active") == 1, 1).otherwise(0)).alias("active_customers"),
        F.sum(F.when(F.col("active") == 0, 1).otherwise(0)).alias("inactive_customers")
    )
    .orderBy(F.desc("inactive_customers"))
)
q6.show()

+------------------+----------------+------------------+
|              city|active_customers|inactive_customers|
+------------------+----------------+------------------+
|          Uluberia|               0|                 1|
|         Najafabad|               0|                 1|
|         Pingxiang|               0|                 1|
|          Xiangfan|               0|                 1|
|        Kumbakonam|               0|                 1|
|       Szkesfehrvr|               0|                 1|
|  Charlotte Amalie|               0|                 1|
|            Kamyin|               0|                 1|
|            Daxian|               0|                 1|
|     Coatzacoalcos|               0|                 1|
|           Wroclaw|               0|                 1|
|            Ktahya|               0|                 1|
|            Amroha|               0|                 1|
|   Southend-on-Sea|               0|                 1|
|           Bat Yam|           

In [63]:
#Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city),
#and that start with the letter “a”. Do the same for cities with a “-” symbol.

rental_hours = (
    rental
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .join(customer, "customer_id")
    .join(address, "address_id")
    .join(city, "city_id")
    .filter(F.col("return_date").isNotNull())
    .withColumn(
        "rental_hours",
        (F.unix_timestamp("return_date") - F.unix_timestamp("rental_date")) / 3600
    )
    .select(
        category["name"].alias("category"),
        city["city"].alias("city"),
        "rental_hours"
    )
)

def top_category_for_cities(df, filter_condition):
    return (
        df
        .filter(filter_condition)
        .groupBy("category")
        .agg(F.sum("rental_hours").alias("total_rental_hours"))
        .orderBy(F.desc("total_rental_hours"))
        .limit(1)
    )

# Города на "a"
q7_a = top_category_for_cities(rental_hours, F.lower(F.col("city")).startswith("a"))
q7_a.show()

# Города с "-"
q7_dash = top_category_for_cities(rental_hours, F.col("city").contains("-"))
q7_dash.show()

+--------+------------------+
|category|total_rental_hours|
+--------+------------------+
|Children|           25834.2|
+--------+------------------+

+--------+------------------+
|category|total_rental_hours|
+--------+------------------+
|   Drama|14556.033333333335|
+--------+------------------+



In [64]:
spark.stop()